In [1]:
# =====================================
# 1. Import Libraries
# =====================================
import pandas as pd
import numpy as np
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.utils import resample

# Download NLTK resources
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\hp5cd\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\hp5cd\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\hp5cd\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

In [2]:
# =====================================
# 2. Load Dataset
# =====================================
df = pd.read_csv("gender_dataset.csv")  # Make sure your CSV path is correct
print("Original dataset shape:", df.shape)
print(df['label'].value_counts())

Original dataset shape: (17880, 2)
label
biased      12691
unbiased     5189
Name: count, dtype: int64


In [3]:
# =====================================
# 3. Text Cleaning Function
# =====================================
def clean_text(text):
    text = text.lower()  # Lowercase
    text = re.sub(r'#URL_\S+', '', text)  # Remove URLs
    text = re.sub(r'[^a-zA-Z\s]', '', text)  # Remove punctuation/numbers
    text = re.sub(r'\s+', ' ', text).strip()  # Remove extra spaces
    return text

df['clean_text'] = df['text'].apply(clean_text)

In [4]:
# =====================================
# 4. Remove Stopwords
# =====================================
stop_words = set(stopwords.words('english'))
df['clean_text'] = df['clean_text'].apply(
    lambda x: ' '.join([w for w in x.split() if w not in stop_words])
)

In [5]:
# =====================================
# 5. Lemmatization
# =====================================
lemmatizer = WordNetLemmatizer()
df['clean_text'] = df['clean_text'].apply(
    lambda x: ' '.join([lemmatizer.lemmatize(w) for w in x.split()])
)

In [6]:
# =====================================
# 6. Handle Imbalanced Classes
# =====================================
df_majority = df[df.label=='biased']
df_minority = df[df.label=='unbiased']

# Upsample minority class
df_minority_upsampled = resample(
    df_minority,
    replace=True,
    n_samples=len(df_majority),
    random_state=42
)

df_balanced = pd.concat([df_majority, df_minority_upsampled])
print("Balanced dataset shape:", df_balanced.shape)
print(df_balanced['label'].value_counts())

Balanced dataset shape: (25382, 3)
label
biased      12691
unbiased    12691
Name: count, dtype: int64


In [7]:
# =====================================
# 7. Save Preprocessed Dataset
# =====================================

# Choose columns to keep
df_balanced_to_save = df_balanced[['clean_text', 'label']]

# Save to CSV
df_balanced_to_save.to_csv("gender_dataset_preprocessed.csv", index=False)

print("Preprocessed dataset saved as 'gender_dataset_preprocessed.csv'")

Preprocessed dataset saved as 'gender_dataset_preprocessed.csv'
